In [1]:
pip install langchain langchain-openai openai chromadb

Note: you may need to restart the kernel to use updated packages.Collecting langchain
  Using cached protobuf-6.33.6-cp310-abi3-win_amd64.whl.metadata (593 bytes)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.2 MB ? eta -:--:--
   --------------------------- ------------ 0.8/1.2 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 1.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/23.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.4 MB ? eta -:--:--
    --------------------------------------- 0.5/23.4 MB 1.3 MB/s eta 0:00:18
   - -------------------------------------- 0.8/23.4 MB 1.3 MB/s eta 0:00:17
   - -------------------------------------- 1.0/23.4 MB 1.4 MB/s eta 0:00:17
   -- ------------------------------------- 1.3/23.4 MB 1.4 MB/s eta 0:00:17
   --- ------------------------------------ 1.8/23.4 MB 1.5 MB/s eta 0:00:15
   --- ---------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires pandas<3,>=1.3.0, but you have pandas 3.0.2 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.6 which is incompatible.


In [10]:
def mock_llm(prompt):
 '''Simple mock LLM for testing without API key'''
 if 'explain' in prompt.lower():
        return 'This association rule suggests customers follow a consistent shopping pattern. Consider creating a product bundle to increase basket size.'
 elif 'recommend' in prompt.lower():
        return 'Based on the transaction data, placing these products near each other or creating a bundle deal could increase sales by 15-25%.'
 else:
        return 'Based on the store data analysis, customers show strong purchasing patterns that can be leveraged for targeted promotions.'

print('LLM setup complete!')
print('Using mock LLM for demo')
print('Replace mock_llm() with real API when available')


LLM setup complete!
Using mock LLM for demo
Replace mock_llm() with real API when available


In [14]:
# Load all data from previous weeks
import pandas as pd
df = pd.read_csv('cognicart/data/cleaned/supermarket_clean.csv')
rules = pd.read_csv('cognicart/data/cleaned/enhanced_rules.csv')
rfm = pd.read_csv('cognicart/data/cleaned/rfm_with_segments.csv')

print('All data loaded!')
print(f'Transactions: {df["TransactionID"].nunique():,}')
print(f'Association rules: {len(rules)}')
print(f'Customer segments: {rfm["Segment"].nunique()}')
print()
print('Top 5 rules by lift:')
print(rules.sort_values('lift', ascending=False)
      [['antecedents','consequents','confidence','lift']].head())


All data loaded!
Transactions: 5,000
Association rules: 1020
Customer segments: 4

Top 5 rules by lift:
                                           antecedents  \
579                    frozenset({'Chips', 'Popcorn'})   
574  frozenset({'Namkeen', 'Chocolate', 'Cold Drink'})   
577                    frozenset({'Chips', 'Namkeen'})   
576  frozenset({'Cold Drink', 'Chocolate', 'Popcorn'})   
582                  frozenset({'Namkeen', 'Popcorn'})   

                                           consequents  confidence      lift  
579  frozenset({'Namkeen', 'Chocolate', 'Cold Drink'})    0.335463  9.266941  
574                    frozenset({'Chips', 'Popcorn'})    0.580110  9.266941  
577  frozenset({'Cold Drink', 'Chocolate', 'Popcorn'})    0.326087  9.159746  
576                    frozenset({'Chips', 'Namkeen'})    0.589888  9.159746  
582    frozenset({'Chips', 'Chocolate', 'Cold Drink'})    0.360825  9.065948  


In [26]:
def explain_rule(antecedent, consequent, confidence, lift):
    '''
    Takes an association rule and explains it
    in plain English using LLM
    '''
    
    prompt = f'''
You are a retail analytics expert.
Explain this association rule in simple business terms:

Rule: If customer buys {antecedent} then they also buy {consequent}
Confidence: {confidence:.0%} (rule is correct this % of time)
Lift: {lift:.2f} (how much more likely than random)

Give a 2-3 sentence business explanation and one recommendation.
Keep it simple and actionable.
'''

    explanation = mock_llm(prompt)
    return explanation
# Test with top rules
print('RULE EXPLANATIONS')
print('=' * 50)

top_rules = rules.sort_values('lift', ascending=False).head(3)

for _, row in top_rules.iterrows():
    print(f'Rule: {row["antecedents"]} -> {row["consequents"]}')
    print(f'Confidence: {float(row["confidence"]):.0%}  Lift: {float(row["lift"]):.2f}')
    explanation = explain_rule(
        row['antecedents'],
        row['consequents'],
        float(row['confidence']),
        float(row['lift'])
    )
    print(f'Explanation: {explanation}')
    print()


RULE EXPLANATIONS
Rule: frozenset({'Chips', 'Popcorn'}) -> frozenset({'Namkeen', 'Chocolate', 'Cold Drink'})
Confidence: 34%  Lift: 9.27
Explanation: This association rule suggests customers follow a consistent shopping pattern. Consider creating a product bundle to increase basket size.

Rule: frozenset({'Namkeen', 'Chocolate', 'Cold Drink'}) -> frozenset({'Chips', 'Popcorn'})
Confidence: 58%  Lift: 9.27
Explanation: This association rule suggests customers follow a consistent shopping pattern. Consider creating a product bundle to increase basket size.

Rule: frozenset({'Chips', 'Namkeen'}) -> frozenset({'Cold Drink', 'Chocolate', 'Popcorn'})
Confidence: 33%  Lift: 9.16
Explanation: This association rule suggests customers follow a consistent shopping pattern. Consider creating a product bundle to increase basket size.



In [32]:
def answer_store_query(question):
    '''
    Answer any question about the store
    using real transaction data + LLM
    '''
    
    # Get relevant data statistics
    total_transactions = df['TransactionID'].nunique()
    total_customers    = df['Customer_ID'].nunique()
    top_products       = df['Product'].value_counts().head(5).index.tolist()
    top_rule           = rules.sort_values('lift', ascending=False).iloc[0]

    # Build context from real data
    context = f'''
Store Statistics:
- Total transactions: {total_transactions}
- Total customers: {total_customers}
- Top 5 products: {top_products}
- Strongest rule: {top_rule['antecedents']} -> {top_rule['consequents']}
  (Confidence: {float(top_rule['confidence']):.0%}, Lift: {float(top_rule['lift']):.2f})
'''

    # Send context + question to LLM
    prompt = f'''
You are a retail analytics assistant.
Use this store data to answer the question:

{context}

Question: {question}

Give a helpful and specific answer based on the data.
'''

    answer = mock_llm(prompt)
    return answer


# Test queries
queries = [
'What are the most popular products in our store?',
'Which products should we bundle together?',
'How can we increase our average basket size?'
]

print('NATURAL LANGUAGE QUERY INTERFACE')
print('=' * 50)
for query in queries:
    print(f'Question: {query}')
    answer = answer_store_query(query)
    print(f'Answer: {answer}')
    print()



NATURAL LANGUAGE QUERY INTERFACE
Question: What are the most popular products in our store?
Answer: Based on the store data analysis, customers show strong purchasing patterns that can be leveraged for targeted promotions.

Question: Which products should we bundle together?
Answer: Based on the store data analysis, customers show strong purchasing patterns that can be leveraged for targeted promotions.

Question: How can we increase our average basket size?
Answer: Based on the store data analysis, customers show strong purchasing patterns that can be leveraged for targeted promotions.



In [40]:
def get_segment_insights(segment_name):
    '''
    Generate business insights for a customer segment
    '''
    
    # Get segment statistics
    segment_data = rfm[rfm['Segment'] == segment_name]
    avg_recency   = segment_data['Recency'].mean()
    avg_frequency = segment_data['Frequency'].mean()
    avg_monetory  = segment_data['Monetory'].mean()
    count         = len(segment_data)

    prompt = f'''
You are a retail business analyst.
Provide insights for this customer segment:

Segment: {segment_name}
Number of customers: {count}
Average days since last visit: {avg_recency:.0f}
Average number of visits: {avg_frequency:.1f}
Average total spending: Rs {avg_monetory:.0f}

Give 3 specific marketing recommendations for this segment.
'''

    insights = mock_llm(prompt)
    return insights
# Generate insights for each segment
print('CUSTOMER SEGMENT INSIGHTS')
print('=' * 50)
for segment in rfm['Segment'].dropna().unique():
    print(f'Segment: {segment}')
    insights = get_segment_insights(segment)
    print(f'Insights: {insights}')
    print()


CUSTOMER SEGMENT INSIGHTS
Segment: Premium Shoppers
Insights: Based on the transaction data, placing these products near each other or creating a bundle deal could increase sales by 15-25%.

Segment: Budget Shoppers
Insights: Based on the transaction data, placing these products near each other or creating a bundle deal could increase sales by 15-25%.

Segment: Regular Shoppers
Insights: Based on the transaction data, placing these products near each other or creating a bundle deal could increase sales by 15-25%.

Segment: Bulk Buyers
Insights: Based on the transaction data, placing these products near each other or creating a bundle deal could increase sales by 15-25%.



In [46]:
def recommend_products(customer_id, top_n=5):
    '''
    Recommend products for a specific customer
    combining MBA rules + BERT similarity + LLM
    '''
    
    # Get customer's purchase history
    customer_data = df[df['Customer_ID'] == customer_id]
    purchased = customer_data['Product'].unique().tolist()

    if not purchased:
        print(f'Customer {customer_id} not found')
        return

    print(f'Customer: {customer_id}')
    print(f'Past purchases: {purchased}')
    print()

    # Get MBA rule recommendations
    mba_recs = []
    for product in purchased:
        prod_rules = rules[rules['antecedents'].str.contains(
            product, na=False
        )]
        for _, rule in prod_rules.iterrows():
            rec = rule['consequents'].strip("frozenset({'})")
            if rec not in purchased:
                mba_recs.append((rec, float(rule['lift'])))

    # Sort by lift
    mba_recs = sorted(mba_recs, key=lambda x: x[1], reverse=True)[:top_n]

    print('MBA Rule Recommendations:')
    for prod, lift in mba_recs:
        print(f'  {prod:20s}  Lift: {lift:.2f}')
    print()

    # Get LLM explanation
    prompt = f'''
Customer bought: {purchased}
MBA recommends: {[p for p,l in mba_recs]}
Write a friendly 1-sentence explanation why these are good recommendations.
'''

    explanation = mock_llm(prompt)
    print(f'Recommendation reason: {explanation}')
# Test recommendations
recommend_products('C0001') 
recommend_products('C0050')

Customer: C0001
Past purchases: ['Cold Drink', 'Chips', 'Sugar', 'Brown Bread', 'Tea', 'Bread', 'Panner', 'Spices', 'Soap', 'Biscuits', 'Tomatosauce', 'Coffee', 'Waterbottle', 'Shampoo', 'Wheat Flour', 'Banana', 'Nuts', 'Apple', 'Oil', 'Cookies', 'Juice', 'Tomato', 'Oats', 'Milk', 'Curd']

MBA Rule Recommendations:
  Chips', 'Popc         Lift: 9.27
  Namkeen', 'Chocolate', 'Cold Drink  Lift: 9.27
  Chips', 'Namk         Lift: 9.16
  Cold Drink', 'Chocolate', 'Popc  Lift: 9.16
  Namkeen', 'Popc       Lift: 9.07

Recommendation reason: Based on the transaction data, placing these products near each other or creating a bundle deal could increase sales by 15-25%.
Customer: C0050
Past purchases: ['Toothpaste', 'Milk', 'Biscuit', 'Biscuits', 'Bread', 'Tea', 'Oats', 'Butter', 'Tomato', 'Spices', 'Shampoo', 'Facewash', 'Juice', 'Chips', 'Nuts', 'Waterbottle', 'Tomatosauce', 'Coffee', 'Poha', 'Potato', 'Curd', 'Sugar', 'Semolina']

MBA Rule Recommendations:
  Namkeen', 'Chocolate', 'Cold Drink

In [48]:
# Generate explanations for top 20 rules
print('Generating rule explanations...')
top20_rules = rules.sort_values('lift', ascending=False).head(20).copy()

explanations = []
for _, row in top20_rules.iterrows():
    explanation = explain_rule(
        row['antecedents'],
        row['consequents'],
        float(row['confidence']),
        float(row['lift'])
    )
    explanations.append(explanation)

top20_rules['LLM_Explanation'] = explanations

# Save results
top20_rules.to_csv(
'cognicart/data/cleaned/rules_with_explanations.csv',
    index=False
)

print('Rules with LLM explanations saved!')
print()
print('Sample explained rule:')
print(top20_rules[['antecedents','consequents','lift','LLM_Explanation']].head(2))


Generating rule explanations...
Rules with LLM explanations saved!

Sample explained rule:
                                           antecedents  \
579                    frozenset({'Chips', 'Popcorn'})   
574  frozenset({'Namkeen', 'Chocolate', 'Cold Drink'})   

                                           consequents      lift  \
579  frozenset({'Namkeen', 'Chocolate', 'Cold Drink'})  9.266941   
574                    frozenset({'Chips', 'Popcorn'})  9.266941   

                                       LLM_Explanation  
579  This association rule suggests customers follo...  
574  This association rule suggests customers follo...  


In [50]:
print('=' * 50)
print('WEEK 7 COMPLETE!')
print('=' * 50)
print()
print('What you built this week:')
print('  Rule explainer using LLM')
print('  Natural language query interface')
print('  Customer segment insights generator')
print('  Product recommendation engine')
print('  Rules saved with LLM explanations')
print()
print('Files saved:')
print('  cognicart/data/cleaned/rules_with_explanations.csv')
print()
print('GitHub push commands:')
print('  git add .')
print('  git commit -m "Week 7 - LLM integration complete"')
print('  git push')
print()
print('Next: Week 8 - Streamlit Dashboard!')


WEEK 7 COMPLETE!

What you built this week:
  Rule explainer using LLM
  Natural language query interface
  Customer segment insights generator
  Product recommendation engine
  Rules saved with LLM explanations

Files saved:
  cognicart/data/cleaned/rules_with_explanations.csv

GitHub push commands:
  git add .
  git commit -m "Week 7 - LLM integration complete"
  git push

Next: Week 8 - Streamlit Dashboard!
